# MIDI BrainAge (Wood 2024) — SIMON batch

**Before running:** Runtime → Change runtime type → **GPU (L4 or T4)**

Runs the MIDIconsortium T1 brain age model on the cleaned SIMON manifest.  
Saves results to Google Drive in the same format as `synthba_predictions.csv`.

**Paper:** Wood et al. (2024). Optimising brain age estimation through transfer learning.  
*Human Brain Mapping*, 45(4), e26625. https://doi.org/10.1002/hbm.26625  
**Repo:** https://github.com/MIDIconsortium/BrainAge

### Notes
- MIDI handles its own skull stripping (HD-BET) and MNI registration internally when `--sequence t1`
- The T1 model **requires** `--skull_strip` flag (raw T1 raises ValueError)
- Project output goes to `./{project_name}_brain_age_output.csv` in the MIDI repo dir
- Each `project_name` can only be used once per session — rerun cell 1 below to clean up

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / pull repos
import os
import subprocess

REPO = '/content/faceage-to-brainage'
MIDI_REPO = '/content/BrainAge'

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kondratevakate/faceage-to-brainage.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase'], check=False)

if not os.path.exists(MIDI_REPO):
    subprocess.run(['git', 'clone', 'https://github.com/MIDIconsortium/BrainAge.git', MIDI_REPO], check=True)
else:
    subprocess.run(['git', '-C', MIDI_REPO, 'pull', '--rebase'], check=False)

os.chdir(MIDI_REPO)
print('Working dir:', os.getcwd())

In [ ]:
# 3. Install MIDI dependencies for modern Colab
import subprocess
import sys

pkgs = [
    "monai",
    "nibabel",
    "xlrd==1.2.0",
    "antspyx",
    "git+https://github.com/MIC-DKFZ/HD-BET",
]

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    capture_output=True,
    text=True,
)
if r.returncode != 0:
    raise RuntimeError("pip install failed:\n" + r.stderr[-8000:])

print("MIDI dependencies installed")


# Verify MIDI model weights exist in repo
import os
from pathlib import Path
t1_model_dir = Path(MIDI_REPO) / 'Models' / 'T1' / 'Skull_stripped'
if not t1_model_dir.exists():
    raise FileNotFoundError(
        f'MIDI T1 skull-stripped models not found at {t1_model_dir}. '
        'Check that the BrainAge repo cloned correctly and includes the Models/ directory.'
    )
print('T1 models found:', [f.name for f in t1_model_dir.glob('*.pt')])

In [ ]:
# 4. Config paths and load manifest
from pathlib import Path
import pandas as pd

DRIVE = '/content/drive/MyDrive'
MIDI_REPO = '/content/BrainAge'

SIMON_ROOT_CANDIDATES = [
    Path(DRIVE) / 'brain_mri' / 't1_only' / 'simon',
    Path(DRIVE) / 'brain_mri' / 'simon',
]
SIMON_ROOT = next((p for p in SIMON_ROOT_CANDIDATES if (p / 'manifest.csv').exists()), None)
if SIMON_ROOT is None:
    print('Searching via rglob (may take 30-60s)...')
    matches = [p.parent for p in Path(DRIVE).rglob('manifest.csv') if p.parent.name.lower() == 'simon']
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one simon/manifest.csv, found {len(matches)}: {matches[:5]}')
    SIMON_ROOT = matches[0]

MANIFEST_SRC = SIMON_ROOT / 'manifest.csv'
IMAGES_DIR = SIMON_ROOT / 'images'
DRIVE_OUT = Path(DRIVE) / 'brain_mri' / 'brainage_results' / 'simon' / 'midi_predictions.csv'
DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_SRC)

def resolve_t1_path(row):
    candidates = []
    if 't1_path' in row.index and pd.notna(row.get('t1_path')):
        raw = str(row['t1_path']).strip()
        candidates += [Path(raw), IMAGES_DIR / Path(raw).name]
    if 't1_filename' in row.index and pd.notna(row.get('t1_filename')):
        candidates.append(IMAGES_DIR / str(row['t1_filename']).strip())
    for c in candidates:
        if c.exists():
            return str(c)
    return str(candidates[0]) if candidates else ''

def clean_key_part(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    t = str(v).strip()
    return '' if t.lower() == 'nan' else t

def build_scan_key(row):
    parts = [
        clean_key_part(row.get('subject_id')),
        clean_key_part(row.get('session_id')),
        clean_key_part(row.get('run')),
        clean_key_part(row.get('acquisition_label')),
        Path(str(row['t1_path'])).name,
    ]
    return '|'.join([p for p in parts if p])

manifest['t1_path'] = manifest.apply(resolve_t1_path, axis=1)
missing = [p for p in manifest['t1_path'] if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f'Missing {len(missing)} inputs. First 5: {missing[:5]}')

manifest['scan_key'] = manifest.apply(build_scan_key, axis=1)

print('SIMON_ROOT:', SIMON_ROOT)
print('Manifest rows:', len(manifest))
display_cols = [c for c in ['subject_id', 'session_id', 'run', 'acquisition_label', 't1_path', 'age', 'scan_key'] if c in manifest.columns]
print(manifest[display_cols].head(3).to_string())

In [ ]:
# 5. GPU check
import subprocess
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

subprocess.run(['nvidia-smi'], check=False)

In [ ]:
import os
import shutil
import importlib
from pathlib import Path

# Patch: replace upstream pre_process.py with our MONAI 1.x-compatible vendor version.
# The cloned BrainAge repo still uses AddChannel which was removed in MONAI 1.0.
_midi_pp = Path("/content/BrainAge/pre_process.py")
_vendor_pp = Path(REPO) / "vendor" / "MIDIBrainAge" / "pre_process.py"
if _vendor_pp.exists():
    shutil.copy2(str(_vendor_pp), str(_midi_pp))
    print(f"[patch] pre_process.py replaced with MONAI 1.x-compatible vendor version")
else:
    print(f"[warn] Vendor pre_process.py not found at {_vendor_pp}")

os.chdir("/content/BrainAge")
import pre_process
importlib.reload(pre_process)

row = manifest.iloc[0]
arr = pre_process.preprocess(
    input_path=str(row["t1_path"]),
    use_gpu=torch.cuda.is_available(),
    skull_strip=True,
    register=True,
    project_name="debug_one2",
)

print(type(arr))
print(None if arr is None else arr.shape)


In [ ]:
# 6. Run MIDI on SIMON with per-scan checkpointing to Drive
import math
import shutil
import subprocess
import sys
import uuid
from pathlib import Path
from typing import List, Dict

import pandas as pd
import torch
from tqdm.auto import tqdm

MIDI_REPO = Path("/content/BrainAge")
DRIVE = Path("/content/drive/MyDrive")
DRIVE_OUT = DRIVE / "brain_mri" / "brainage_results" / "simon" / "midi_predictions.csv"
TMP_OUT = Path("/tmp/midi_simon_predictions.csv")
TMP_INPUT = Path("/tmp/midi_one_scan.csv")

USE_GPU = torch.cuda.is_available()

def load_existing(path: Path) -> List[Dict]:
    if path.exists():
        return pd.read_csv(path).to_dict("records")
    return []

records = load_existing(DRIVE_OUT) if DRIVE_OUT.exists() else load_existing(TMP_OUT)
done = {str(r["scan_key"]) for r in records if str(r.get("status", "")) == "ok"}

print("Existing rows:", len(records))
print("Already done:", len(done))
print("Drive out:", DRIVE_OUT)

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="MIDI SIMON"):
    scan_key = str(row["scan_key"])
    if scan_key in done:
        continue

    one = pd.DataFrame([{
        "ID": scan_key,
        "file_name": str(row["t1_path"]),
        "Age": float(row["age"]) if not pd.isna(row["age"]) else "",
    }])
    one.to_csv(TMP_INPUT, index=False)

    project_name = f"midi_{uuid.uuid4().hex[:8]}"
    project_dir = MIDI_REPO / project_name
    batch_csv = MIDI_REPO / f"{project_name}_brain_age_output.csv"

    record = {
        "scan_key": scan_key,
        "subject_id": str(row.get("subject_id", "")),
        "session_id": row.get("session_id", ""),
        "run": row.get("run", ""),
        "acquisition_label": row.get("acquisition_label", ""),
        "chron_age": float(row["age"]) if not pd.isna(row["age"]) else float("nan"),
        "predicted_age": float("nan"),
        "brain_age_gap": float("nan"),
        "model_name": "MIDI_Wood2024",
        "status": "error",
        "error": "",
        "input_path": str(row["t1_path"]),
    }

    cmd = [
        sys.executable,
        str(MIDI_REPO / "run_inference.py"),
        "--project_name", project_name,
        "--csv_file", str(TMP_INPUT),
        "--sequence", "t1",
        "--ensemble",
        "--skull_strip",
        "--return_metrics",
    ]
    if USE_GPU:
        cmd.append("--gpu")

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            cwd=MIDI_REPO,
            timeout=1800,
        )

        if result.returncode != 0:
            record["error"] = (result.stderr or result.stdout)[-3000:]
        elif not batch_csv.exists():
            record["error"] = f"Expected output not found: {batch_csv}"
        else:
            out = pd.read_csv(batch_csv)
            pred_col = "Predicted_age (years)"
            if pred_col not in out.columns or out.empty:
                record["error"] = f"Unexpected MIDI output columns: {list(out.columns)}"
            else:
                pred = float(out[pred_col].iloc[0])
                record["predicted_age"] = pred
                if not math.isnan(record["chron_age"]):
                    record["brain_age_gap"] = pred - record["chron_age"]
                record["status"] = "ok"
                done.add(scan_key)

    except Exception as e:
        record["error"] = str(e)

    records.append(record)
    pd.DataFrame(records).to_csv(TMP_OUT, index=False)
    DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TMP_OUT, DRIVE_OUT)

    shutil.rmtree(project_dir, ignore_errors=True)
    if batch_csv.exists():
        batch_csv.unlink(missing_ok=True)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Done")
print("Saved to:", DRIVE_OUT)


In [ ]:
# 7. Verify results saved by Cell 7
import pandas as pd
from pathlib import Path

DRIVE = "/content/drive/MyDrive"
DRIVE_OUT = Path(DRIVE) / "brain_mri" / "brainage_results" / "simon" / "midi_predictions.csv"

if not DRIVE_OUT.exists():
    raise FileNotFoundError(
        "DRIVE_OUT not found: " + str(DRIVE_OUT) + ". Run Cell 7 (inference loop) first."
    )

df = pd.read_csv(DRIVE_OUT)
ok = df[df["status"] == "ok"]
fail = df[df["status"] != "ok"]
print(f"Loaded {len(df)} records from {DRIVE_OUT}")
print(f"  success: {len(ok)}, failed/missing: {len(fail)}")
if not ok.empty:
    print(ok[["subject_id", "session_id", "run", "chron_age", "predicted_age", "brain_age_gap"]].head(10).to_string())


In [ ]:
# 8. Summary (run after reopening if runtime was disconnected)
import numpy as np
import pandas as pd
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
DRIVE_OUT = Path(DRIVE) / 'brain_mri' / 'brainage_results' / 'simon' / 'midi_predictions.csv'

df = pd.read_csv(DRIVE_OUT)
ok = df[df['status'] == 'ok'].copy()

print('rows total:', len(df))
print('rows ok:', len(ok))

if len(ok):
    err = ok['predicted_age'] - ok['chron_age']
    print('MAE:', float(err.abs().mean()))
    print('RMSE:', float(np.sqrt(max((err ** 2).mean(), 0.0))))
    print('Bias (mean gap):', float(err.mean()))
    print('Predicted age std:', float(ok['predicted_age'].std()))
    print(ok[['subject_id', 'session_id', 'run', 'chron_age', 'predicted_age', 'brain_age_gap']].head(10).to_string())